In [1]:
import os
from dotenv import load_dotenv
from qiskit_ibm_runtime import QiskitRuntimeService

# 1. Load the hidden .env file
load_dotenv()

# 2. Grab the API key from your environment variables
API_KEY = os.getenv("IBM_QUANTUM_TOKEN")

try:
    # 3. Initialize using your safely loaded token
    if not API_KEY:
        raise ValueError("API Key missing! Check your .env file.")
        
    service = QiskitRuntimeService(channel="ibm_quantum_platform", token=API_KEY)
    print("Successfully authenticated with IBM Quantum!")
    
    backend = service.backend("ibm_kingston")
    print(f"Connected to backend: {backend.name} ({backend.num_qubits} qubits available)")
except Exception as e:
    print(f"Authentication failed. Error: {e}")

qiskit_runtime_service._discover_account:WARNING:2026-06-18 10:01:10,816: Loading account with the given token. A saved account will not be used.
qiskit_runtime_service.__init__:WARNING:2026-06-18 10:01:15,506: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-06-18 10:01:15,507: Using instance: open-instance, plan: open


Successfully authenticated with IBM Quantum!
Connected to backend: ibm_kingston (156 qubits available)


In [3]:
# Cell 2: Define constraints and build the quantum circuit
from qiskit import QuantumCircuit

# Setup constraints matching your exact requirements
BITS_PER_SEQUENCE = 1000000   # 1 Million bits per file
TOTAL_SEQUENCES = 1000        # 1000 individual sequences

# Optimize layout for ibm_kingston (156 qubits available)
# We will use 100 qubits in parallel to make the math perfectly clean
NUM_QUBITS = 100 
SHOTS_PER_JOB = BITS_PER_SEQUENCE // NUM_QUBITS # 10,000 shots per batch

print(f"Configuration Plan:")
print(f"- Target: {TOTAL_SEQUENCES} files of {BITS_PER_SEQUENCE} bits each.")
print(f"- Per Hardware Run: {NUM_QUBITS} qubits parallelized x {SHOTS_PER_JOB} shots = {NUM_QUBITS * SHOTS_PER_JOB} bits generated.")

# Construct the massive parallel QRNG circuit
qc = QuantumCircuit(NUM_QUBITS)

# Apply a Hadamard gate to all 100 qubits to induce 50/50 superposition
qc.h(range(NUM_QUBITS))

# Measure all qubits simultaneously into a classical register
qc.measure_all()

print("\nQuantum circuit successfully initialized and structured!")

Configuration Plan:
- Target: 1000 files of 1000000 bits each.
- Per Hardware Run: 100 qubits parallelized x 10000 shots = 1000000 bits generated.

Quantum circuit successfully initialized and structured!


In [7]:
# Cell 3: Transpile and execute the circuit using SamplerV2
import os
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2

# Define target folder to save your sequence streams
output_dir = "quantum_random_sequences"
os.makedirs(output_dir, exist_ok=True)

print(f"Initializing optimization pass for {backend.name}...")

# 1. Initialize the Staged Pass Manager for hardware compilation
# Optimization level 1 is ideal for simple mapping without losing structure
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)

print("Transpiling abstract Hadamard circuit to target native hardware instructions...")
# This translates 'h' gates into physical hardware-supported microwave pulses
transpiled_qc = pm.run(qc)
print("Transpilation successful! Circuit optimized for ibm_kingston topology.")

# 2. Establish the Sampler Runtime mode
sampler = SamplerV2(mode=backend)

# Track Sequence #1 
sequence_id = 1
file_path = os.path.join(output_dir, f"sequence_{sequence_id:04d}.txt")

print(f"\nSubmitting compiled job for Sequence {sequence_id}/{TOTAL_SEQUENCES}...")
print(f"Requesting {SHOTS_PER_JOB} shots across the hardware-mapped qubits...")

try:
    # Submit the physical transpiled circuit execution batch
    job = sampler.run([(transpiled_qc, None, SHOTS_PER_JOB)])
    print(f"Job submitted successfully! Job ID: {job.job_id()}")
    print("Waiting for hardware execution to finish...")
    
    # Retrieve results from the IBM Quantum Cloud
    result = job.result()
    
    # Access the classical register outputs
    pub_result = result[0]
    
    # Look up the system-assigned register name from the transpiled circuit
    register_name = transpiled_qc.cregs[0].name
    data_bitstrings = pub_result.data[register_name].get_bitstrings()
    
    # Concatenate all parallel shot outputs into one continuous stream
    full_bit_stream = "".join(data_bitstrings)
    
    # Verify the exact bit budget constraint matches your criteria
    generated_len = len(full_bit_stream)
    print(f"\nData extraction complete! Generated string length: {generated_len} bits.")
    
    # Write the raw bits directly as ASCII text characters
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(full_bit_stream)
        
    print(f"🎉 Success! Sequence saved to: {file_path}")
    print(f"File Size on Disk: {os.path.getsize(file_path)} bytes (~1 MB)")

except Exception as e:
    print(f"\nAn error occurred during runtime execution: {e}")

Initializing optimization pass for ibm_kingston...
Transpiling abstract Hadamard circuit to target native hardware instructions...
Transpilation successful! Circuit optimized for ibm_kingston topology.

Submitting compiled job for Sequence 1/1000...
Requesting 10000 shots across the hardware-mapped qubits...
Job submitted successfully! Job ID: d8p20puhm1is739mu260
Waiting for hardware execution to finish...

Data extraction complete! Generated string length: 1000000 bits.
🎉 Success! Sequence saved to: quantum_random_sequences/sequence_0001.txt
File Size on Disk: 1000000 bytes (~1 MB)


In [9]:
# Cell 4: High-speed batch processing to fit within your 10-minute window
import os
import time
from qiskit_ibm_runtime import SamplerV2

print("⚡ Initializing High-Speed Batch Pipeline...")

# Configuration for batching
BATCH_SIZE = 50  # Number of sequences to bundle into a single hardware job submission
total_sequences = TOTAL_SEQUENCES # 1000
output_dir = "quantum_random_sequences"

# Create a list of all sequence IDs we need to generate
sequences_to_run = [sid for sid in range(1, total_sequences + 1) 
                    if not os.path.exists(os.path.join(output_dir, f"sequence_{sid:04d}.txt"))]

print(f"Total sequences remaining to generate: {len(sequences_to_run)}")

# Establish your runtime sampler instance
sampler = SamplerV2(mode=backend)

# Process the sequences in large parallel chunks
for i in range(0, len(sequences_to_run), BATCH_SIZE):
    batch_ids = sequences_to_run[i:i + BATCH_SIZE]
    print(f"\nPackaging batch for sequences: {batch_ids[0]} to {batch_ids[-1]}...")
    
    # Bundle identical transpiled circuits for this batch execution segment
    # Each item in the bundle is a tuple: (circuit, positional_parameters, shots)
    circuit_bundle = [(transpiled_qc, None, SHOTS_PER_JOB) for _ in batch_ids]
    
    start_time = time.time()
    try:
        print(f"🚀 Submitting batch job of {len(circuit_bundle)} sequences to ibm_kingston...")
        job = sampler.run(circuit_bundle)
        print(f"Job ID allocated: {job.job_id()}. Awaiting hardware execution array...")
        
        # Pull down the collective result for the entire batch at once
        result = job.result()
        register_name = transpiled_qc.cregs[0].name
        
        # Iterate through the returned results and map them back to your files instantly
        for idx, seq_id in enumerate(batch_ids):
            pub_result = result[idx]
            data_bitstrings = pub_result.data[register_name].get_bitstrings()
            full_bit_stream = "".join(data_bitstrings)
            
            # File save pipeline
            file_path = os.path.join(output_dir, f"sequence_{seq_id:04d}.txt")
            with open(file_path, "w", encoding="utf-8") as f:
                f.write(full_bit_stream)
                
        elapsed = time.time() - start_time
        print(f"✅ Batch completed! Saved {len(batch_ids)} files in {elapsed:.2f} seconds.")
        
    except Exception as e:
        print(f"❌ Error during batch execution: {e}")
        print("Breaking loop to preserve configuration state.")
        break

print("\n🏁 Fast batching pipeline sequence closed.")

⚡ Initializing High-Speed Batch Pipeline...
Total sequences remaining to generate: 987

Packaging batch for sequences: 14 to 63...
🚀 Submitting batch job of 50 sequences to ibm_kingston...
Job ID allocated: d8p22g0q90bc73e776ug. Awaiting hardware execution array...
✅ Batch completed! Saved 50 files in 157.06 seconds.

Packaging batch for sequences: 64 to 113...
🚀 Submitting batch job of 50 sequences to ibm_kingston...
Job ID allocated: d8p23n29m3dc738p9u00. Awaiting hardware execution array...
✅ Batch completed! Saved 50 files in 158.96 seconds.

Packaging batch for sequences: 114 to 163...
🚀 Submitting batch job of 50 sequences to ibm_kingston...
Job ID allocated: d8p24ugq90bc73e77b20. Awaiting hardware execution array...
✅ Batch completed! Saved 50 files in 175.15 seconds.

Packaging batch for sequences: 164 to 213...
🚀 Submitting batch job of 50 sequences to ibm_kingston...
Job ID allocated: d8p26agq90bc73e77de0. Awaiting hardware execution array...
✅ Batch completed! Saved 50 files

/opt/homebrew/Caskroom/miniforge/base/envs/quantum_crypto/lib/python3.11/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py:1224: UserWarning: This instance has met its usage limit. Workloads will not run until time ismade available. Check https://quantum.cloud.ibm.com/instances/crn%3Av1%3Abluemix%3Apublic%3Aquantum-computing%3Aus-east%3Aa%2F0ded06f11aeb4cfe84b08a018c4e3bba%3Ac473bb7b-1760-4362-9310-3405fff9d650%3A%3A for more details.
  warnings.warn(


Job ID allocated: d8p27hmhm1is739muc70. Awaiting hardware execution array...


KeyboardInterrupt: 

In [10]:
import os
import glob
import shutil
from datetime import datetime

# Source directory
source_dir = "quantum_random_sequences"
files = sorted(glob.glob(os.path.join(source_dir, "sequence_*.txt")))

# Target directories for sorting
ibm_dir = os.path.join(source_dir, "pure_ibm_quantum")
local_dir = os.path.join(source_dir, "local_plan_b")

os.makedirs(ibm_dir, exist_ok=True)
os.makedirs(local_dir, exist_ok=True)

print("🔍 Analyzing file timestamps to verify origins...\n")

# Group files by their exact creation minute to detect the fast Plan B dump
timestamp_counts = {}
file_timestamps = {}

for f_path in files:
    stat = os.stat(f_path)
    # Using modification time as the birth/write timestamp
    timestamp = stat.st_mtime
    dt = datetime.fromtimestamp(timestamp)
    minute_key = dt.strftime("%Y-%m-%d %H:%M")
    
    file_timestamps[f_path] = (timestamp, minute_key)
    timestamp_counts[minute_key] = timestamp_counts.get(minute_key, 0) + 1

ibm_count = 0
plan_b_count = 0

for f_path in files:
    timestamp, minute_key = file_timestamps[f_path]
    f_name = os.path.basename(f_path)
    
    # If a single minute has a massive spike of files (e.g., > 10 files written at once),
    # it was unmistakably generated by the lightning-fast local Plan B loop.
    if timestamp_counts[minute_key] > 10:
        shutil.move(f_path, os.path.join(local_dir, f_name))
        plan_b_count += 1
    else:
        shutil.move(f_path, os.path.join(ibm_dir, f_name))
        ibm_count += 1

print("Sorting complete!")
print(f"📥 Moved to '{ibm_dir}': {ibm_count} files (Pure IBM Quantum)")
print(f"💻 Moved to '{local_dir}': {plan_b_count} files (Plan B Local Fallback)")

🔍 Analyzing file timestamps to verify origins...

Sorting complete!
📥 Moved to 'quantum_random_sequences/pure_ibm_quantum': 13 files (Pure IBM Quantum)
💻 Moved to 'quantum_random_sequences/local_plan_b': 987 files (Plan B Local Fallback)


In [11]:
import os
import glob

# Path to your pure quantum files
ibm_files = sorted(glob.glob("quantum_random_sequences/pure_ibm_quantum/sequence_*.txt"))

stitched_stream = ""
for f_path in ibm_files:
    with open(f_path, "r", encoding="utf-8") as f:
        stitched_stream += f.read().strip()

# Save as a single master file
output_path = "quantum_random_sequences/pure_ibm_quantum_master_13M.txt"
with open(output_path, "w", encoding="utf-8") as f:
    f.write(stitched_stream)

print(f"✅ Successfully stitched {len(ibm_files)} files!")
print(f"Total size of master bitstream: {len(stitched_stream)} bits.")

✅ Successfully stitched 13 files!
Total size of master bitstream: 13000000 bits.
